# 017-gray-logit-refiner

# Imports

In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict
from functools import partial

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.plotting.segmentation import colour_labels, fixed_broken
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate, load_baseline_model, load_val_sets,
                                      lraspp_outputs, results_table, run_model, upsample_logits)
from uavloc.segmentation.latency import model_latency, refiner_latency, standard_tile
from uavloc.segmentation.losses import ce_dice_loss
from uavloc.segmentation.refine.refinement_cnn import (TrainConfig, build_crop_bank, is_finite, n_parameters,
                                                       refine_fn, train_refiner, val_crop_bank)
from uavloc.segmentation.refine.residual_refiner import ResidualRefiner, ResidualRefinerConfig
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed, tensorboard_writer

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "017-gray-logit-refiner"
NOTE = ""
LOG_RUN = True
MODE = "refiner"                  # "refiner" | "baseline"
IMAGE = "gray"                    # "gray" | "rgb"
SCORES = "logits"                 # "logits" | "log_probs"
INPUT_NORM = False
POINTWISE_SKIP = False
SEED = 42
WIDTH = 32
DILATIONS = [1, 2, 4]
LAMBDA_DICE = 1.0
N_TRAIN_CROPS = 512
N_HOLDOUT_CROPS = 64
HOLDOUT_SEED = 123
BANK_CROP_PX = 512
BANK_BATCH = 8
VAL_CROP_PX = 512
RARE_CROP_P = 0.3
GAP_PX = 64
STEPS = 3000
BATCH = 16
CROP_PX = 256
LR = 3e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
LOG_EVERY = 250
TRAIN_DEVICE = "mps"
EVAL_DEVICE = "cpu"
PROBE_RESULTS = "pocs/mobilenet-lraspp-speed/results-lraspp.json"
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_WARMUP = 5
LATENCY_RUNS = 30
LATENCY_THREADS = 4
ZOOM_PX = 256
ZOOMS = [["birky-strip-google", 0, 64, 256, "buildings"], ["birky-strip-google", 0, 192, 2304, "track"],
         ["bushcha", 0, 1664, 1472, "buildings"], ["bushcha", 0, 1728, 768, "track"]]
REGION_METRICS = ["miou", "mdice", "biou", "bf1_1px", "bf1_2px"]
PER_CLASS_METRICS = ["iou", "dice", "biou"]

# Data

In [ ]:
set_seed(SEED)
train_device = torch.device(TRAIN_DEVICE if TRAIN_DEVICE != "mps" or torch.backends.mps.is_available() else "cpu")
as_logits = SCORES == "logits"
refiner_config = ResidualRefinerConfig(width=WIDTH, dilations=tuple(DILATIONS), image=IMAGE, input_norm=INPUT_NORM,
                                       pointwise_skip=POINTWISE_SKIP)
train_config = TrainConfig(steps=STEPS, batch=BATCH, crop_px=CROP_PX, lr=LR, weight_decay=WEIGHT_DECAY,
                           warmup_frac=WARMUP_FRAC, log_every=LOG_EVERY)
loss_fn = partial(ce_dice_loss, dice_weight=LAMBDA_DICE)
train_device

In [ ]:
val_sets = load_val_sets()
{name: len(images) for name, images in val_sets.items()}

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / "ua-village-annotated", DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
probe_splits = json.loads((REPO_ROOT / PROBE_RESULTS).read_text())["splits"]["sat_train"]
assert [(s.loc, s.date) for s in sat_train] == [tuple(p) for p in probe_splits]
[(s.loc.split("-")[0], s.date) for s in sat_train]

# Model

In [ ]:
segmenter = load_baseline_model(EVAL_DEVICE)

In [ ]:
cases = run_model(segmenter, val_sets, EVAL_DEVICE)

In [ ]:
refiner = ResidualRefiner(refiner_config)
n_parameters(refiner), refiner_config.receptive_radius_px

# Run

In [ ]:
baseline = evaluate(bilinear_argmax, cases)
results_table(baseline).round(4)

In [ ]:
def train_banks() -> tuple:
    """Training and hold-out crops drawn by LR-ASPP's own sampler, with the frozen model's stride-8 logits."""
    sampler = vs.SatelliteSampler(sat_train, BANK_CROP_PX, RARE_CROP_P)
    bank_model = load_baseline_model(train_device)
    train = build_crop_bank(bank_model, sampler, N_TRAIN_CROPS, BANK_BATCH, SEED, train_device)
    holdout = build_crop_bank(bank_model, sampler, N_HOLDOUT_CROPS, BANK_BATCH, HOLDOUT_SEED, train_device)
    return train, holdout

In [ ]:
history, train_minutes = [], 0.0
writer = tensorboard_writer(EXP)
if MODE == "refiner":
    start = time.time()
    train_bank, holdout_bank = train_banks()
    bank_minutes = (time.time() - start) / 60
    val_bank = val_crop_bank(cases, VAL_CROP_PX)
    history = train_refiner(refiner, train_bank, holdout_bank, train_config, SEED, train_device, loss_fn, as_logits,
                            val=val_bank, writer=writer)
    train_minutes = (time.time() - start) / 60 - bank_minutes
    refiner = refiner.cpu().eval()
pd.DataFrame(history).round(4)

In [ ]:
predict = refine_fn(refiner, EVAL_DEVICE, as_logits) if MODE == "refiner" else bilinear_argmax
results = evaluate(predict, cases)
results_table(results).round(4)

In [ ]:
def delta_table(metrics: list[str]) -> pd.DataFrame:
    """Refined minus baseline for each val set and the mean."""
    return pd.DataFrame({n: {k: results[n][k] - baseline[n][k] for k in metrics}
                         for n in [*VAL_SETS, MEAN_KEY]}).T

In [ ]:
delta_table(REGION_METRICS).round(4)

In [ ]:
def per_class_frame(metric: str) -> pd.DataFrame:
    """`<metric>_per_class` for every val set: rows (set, baseline | refined), one column per class."""
    return pd.concat({n: pd.DataFrame({"baseline": baseline[n][f"{metric}_per_class"],
                                       "refined": results[n][f"{metric}_per_class"]}).T for n in VAL_SETS})

In [ ]:
per_class = {metric: per_class_frame(metric) for metric in PER_CLASS_METRICS}
pd.concat(per_class, axis=1).round(3)

In [ ]:
latency_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(segmenter, latency_tile, EVAL_DEVICE)
timing = dict(warmup=LATENCY_WARMUP, runs=LATENCY_RUNS, threads=LATENCY_THREADS)
model_before = model_latency(segmenter, latency_tile, **timing)
refiner_time = refiner_latency(predict, latency_output, latency_tile, **timing)
model_after = model_latency(segmenter, latency_tile, **timing)
model_ms = (model_before.median_ms + model_after.median_ms) / 2
latency_ratio = refiner_time.median_ms / model_ms
{"model_ms": [model_before.median_ms, model_after.median_ms], "refiner_ms": refiner_time.median_ms,
 "ratio": latency_ratio}

# Plots

In [ ]:
fig_curves, (ax_ce, ax_loss) = plt.subplots(1, 2, figsize=(12, 3.8))
if history:
    frame = pd.DataFrame(history)
    ax_ce.plot(frame["step"], frame["train_ce"], "-o", label="train")
    ax_ce.plot(frame["step"], frame["holdout_ce"], "-s", label="hold-out (train images)")
    ax_ce.plot(frame["step"], frame["val_ce"], "-^", label="val")
    ax_ce.plot(frame["step"], frame["val_ce_identity"], "--", label="val, LR-ASPP alone")
    ax_loss.plot(frame["step"], frame["train_loss"], "-o", label="train")
    ax_loss.plot(frame["step"], frame["val_loss"], "-^", label="val")
    ax_loss.plot(frame["step"], frame["val_loss_identity"], "--", label="val, LR-ASPP alone")
ax_ce.set_title("cross-entropy"); ax_loss.set_title(f"objective: CE + {LAMBDA_DICE}·Dice")
for ax in (ax_ce, ax_loss):
    ax.set_xlabel("step"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_curves.suptitle(f"{IMAGE} + {SCORES}, seed={SEED}")
fig_curves.tight_layout()

In [ ]:
def metric_bars(ax: plt.Axes, metric: str) -> None:
    """Baseline vs refined `metric` per val set and the mean."""
    names, width = [*VAL_SETS, MEAN_KEY], 0.38
    pos = np.arange(len(names))
    for shift, (label, res) in zip((-width / 2, width / 2), (("LR-ASPP", baseline), ("+ refiner", results))):
        bars = ax.bar(pos + shift, [res[n][metric] for n in names], width, label=label)
        ax.bar_label(bars, fmt="%.3f", fontsize=7)
    ax.set_xticks(pos, names, rotation=20, fontsize=8); ax.set_title(metric); ax.grid(axis="y", alpha=.3)

In [ ]:
fig_metrics, axes = plt.subplots(1, len(REGION_METRICS), figsize=(4 * len(REGION_METRICS), 3.8))
for ax, metric in zip(axes, REGION_METRICS):
    metric_bars(ax, metric)
axes[0].legend(fontsize=8)
fig_metrics.tight_layout()

In [ ]:
def class_delta_bars(ax: plt.Axes, metric: str) -> None:
    """Refined minus baseline per class, one bar group per val set."""
    frame = pd.DataFrame({n: per_class[metric].loc[(n, "refined")] - per_class[metric].loc[(n, "baseline")]
                          for n in VAL_SETS})
    frame.plot.bar(ax=ax, width=.8, legend=False)
    ax.axhline(0, color="k", lw=.8); ax.set_title(f"Δ {metric} per class"); ax.grid(axis="y", alpha=.3)
    ax.tick_params(axis="x", labelrotation=20, labelsize=8)

In [ ]:
fig_classes, axes = plt.subplots(1, len(PER_CLASS_METRICS), figsize=(5 * len(PER_CLASS_METRICS), 3.8))
for ax, metric in zip(axes, PER_CLASS_METRICS):
    class_delta_bars(ax, metric)
axes[0].legend(fontsize=8)
fig_classes.tight_layout()

In [ ]:
@torch.no_grad()
def delta_magnitude(case, top: int, left: int) -> np.ndarray:
    """max_c |ΔZ_c - mean_c ΔZ| over a `ZOOM_PX` window (a shift shared by all classes leaves the softmax
    unchanged), computed with a receptive-radius margin so edges are exact."""
    if MODE != "refiner":
        return np.zeros((ZOOM_PX, ZOOM_PX))
    h, w = case.image.rgb.shape[:2]
    r = refiner_config.receptive_radius_px
    t0, l0, t1, l1 = max(top - r, 0), max(left - r, 0), min(top + ZOOM_PX + r, h), min(left + ZOOM_PX + r, w)
    z = torch.from_numpy(upsample_logits(case.output.logits, (h, w))[:, t0:t1, l0:l1].copy())[None]
    x = torch.from_numpy(normalise(case.image.rgb[t0:t1, l0:l1]).transpose(2, 0, 1).copy())[None]
    d = refiner.delta(x, z)[0]
    d = (d - d.mean(0)).abs().amax(0).numpy()
    return d[top - t0:top - t0 + ZOOM_PX, left - l0:left - l0 + ZOOM_PX]

In [ ]:
def zoom_row(axes: np.ndarray, zoom: list) -> None:
    """RGB | label | LR-ASPP | refined | fixed/broken | class-relative |ΔZ| for one `ZOOM_PX` window of a val image."""
    name, index, top, left, what = zoom
    case, win = cases[name][index], np.s_[top:top + ZOOM_PX, left:left + ZOOM_PX]
    label, before, after = case.image.label[win], bilinear_argmax(case.output, case.image.rgb)[win], \
        predict(case.output, case.image.rgb)[win]
    panels = [(case.image.rgb[win], f"{name} {case.image.date} ({what})"), (colour_labels(label), "label"),
              (colour_labels(before), "LR-ASPP bilinear"), (colour_labels(after), f"{MODE}: {IMAGE} + {SCORES}"),
              (fixed_broken(label, before, after), "fixed (green) / broken (red)")]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")
    im = axes[-1].imshow(delta_magnitude(case, top, left), cmap="magma"); axes[-1].axis("off")
    axes[-1].set_title("max_c |ΔZ_c − mean ΔZ|", fontsize=9); plt.colorbar(im, ax=axes[-1], fraction=.046)

In [ ]:
fig_zoom, axes = plt.subplots(len(ZOOMS), 6, figsize=(19, 3.3 * len(ZOOMS)))
for row, zoom in zip(axes, ZOOMS):
    zoom_row(row, zoom)
fig_zoom.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
                loc="lower center", ncol=len(CLASSES))
fig_zoom.tight_layout(rect=(0, .03, 1, 1))

# Log

In [ ]:
def run_metrics() -> dict:
    """Primary metric first (report.py takes the first scalar), then per set, deltas, per class, latency."""
    mean = results[MEAN_KEY]
    metrics = {"miou_mean": mean["miou"], "mdice_mean": mean["mdice"], "boundary_iou_mean": mean["biou"],
               "bf1_1px_mean": mean["bf1_1px"], "bf1_2px_mean": mean["bf1_2px"]}
    metrics.update({f"{k}_{n}": v for n in VAL_SETS for k, v in results[n].items() if not isinstance(v, dict)})
    metrics.update({f"delta_{k}_{n}": v for n, row in delta_table(REGION_METRICS).iterrows() for k, v in row.items()})
    metrics.update({f"{m}_{c}_{n}": v for m in PER_CLASS_METRICS for n in VAL_SETS
                    for c, v in results[n][f"{m}_per_class"].items()})
    metrics.update({"refiner_ms_median": refiner_time.median_ms, "refiner_ms_p10": refiner_time.p10_ms,
                    "refiner_ms_p90": refiner_time.p90_ms, "model_ms_median": model_ms,
                    "refiner_to_model_ratio": latency_ratio, "n_parameters": n_parameters(refiner),
                    "train_minutes": train_minutes})
    if history:
        last = history[-1]
        metrics.update({k: last[k] for k in ("holdout_ce", "holdout_ce_identity", "val_ce", "val_ce_identity",
                                             "val_loss", "val_loss_identity")})
        metrics.update({"val_loss_min_step": min(history, key=lambda h: h["val_loss"])["step"], "history": history})
    return metrics

In [ ]:
metrics = run_metrics()
RUN_CONFIG = {"mode": MODE, "refiner": asdict(refiner_config), "scores": SCORES, "lambda_dice": LAMBDA_DICE,
              "train": asdict(train_config), "n_train_crops": N_TRAIN_CROPS, "n_holdout_crops": N_HOLDOUT_CROPS,
              "holdout_seed": HOLDOUT_SEED, "bank_crop_px": BANK_CROP_PX, "val_crop_px": VAL_CROP_PX, "rare_crop_p": RARE_CROP_P,
              "gap_px": GAP_PX, "segmenter": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
              "eval_device": EVAL_DEVICE, "latency_threads": LATENCY_THREADS}
{k: round(v, 4) for k, v in metrics.items() if isinstance(v, float) and ("mean" in k or "ms" in k or "ratio" in k)}

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
variant = "".join(["-norm" * INPUT_NORM, "-pwskip" * POINTWISE_SKIP])
checkpoint = artifacts / f"refiner-{IMAGE}-{SCORES}{variant}-seed{SEED}.pt"
if MODE == "refiner":
    torch.save(refiner.state_dict(), checkpoint)
status = "ok" if is_finite(history) else "failed"
figures = [fig_curves, fig_metrics, fig_classes, fig_zoom]
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures, status=status, seed=SEED,
              checkpoint=str(checkpoint.relative_to(REPO_ROOT)) if MODE == "refiner" else None,
              device=str(train_device) if MODE == "refiner" else EVAL_DEVICE, writer=writer) if LOG_RUN else {"run": "dry"}
writer.close()
run["run"]